# 08 - Backtesting & Portfolio Allocation
Replays historical signals against actual outcomes: **AlphaForge Strategy vs Buy & Hold vs Market Index** on CAGR, Sharpe, max drawdown, win rate, total return, volatility. Then builds a portfolio allocation from expected return/risk/correlation.

> Logic lives in `src/portfolio/` (backtester.py, optimizer.py) + `src/engine/risk_analytics.py`.
> Signals act the *next* session (no lookahead); costs charged on position changes.

## 1. Backtest NVDA over the test window

In [ ]:
import pandas as pd

import src.api.service as svc

bt = svc.backtest_payload('NVDA')
compare = pd.DataFrame(bt['comparison']).T
compare

## 2. Interpretation
- **alphaforge_strategy**: model direction probability >0.55 long, <0.45 short, else flat
- **buy_hold**: always-long NVDA
- **market_index**: S&P 500 benchmark

In [ ]:
print('trades taken:', bt.get('n_trades'))
print('strategy CAGR vs buy&hold:', compare.loc['alphaforge_strategy','cagr'], 'vs', compare.loc['buy_hold','cagr'])

## 3. Portfolio allocation (mean-variance max-Sharpe, inverse-vol fallback)

In [ ]:
p = svc.portfolio_payload(['AAPL','MSFT','NVDA','AMZN'])
print('method:', p['method'])
print(p['allocations'])
print('stats:', p['stats'])
print('warnings:', p['risk_warnings'])

## 4. Alternative allocation methods

In [ ]:
from src.portfolio.optimizer import recommend_allocation
import src.api.service as _s

prices = pd.DataFrame({s: _s.load_feature_table(s)['Close'] for s in ['AAPL','MSFT','NVDA','AMZN']}).dropna()
for m in ('inverse_vol','risk_parity','mean_variance'):
    out = recommend_allocation(prices, method=m)
    print(f"{m:>14}: {out['allocations']} sharpe={out['stats']['sharpe']}")